# Soliton Binary Merger

Philip Mocz (2025)

Collide two solitons with opposite phases with jaxion!

# 0. Setup

If you are running in Google Colab and would like to use GPUs, go to:

  Edit --> Notebook settings --> select [T4 GPU] as the Hardware Accelerator

and install:

In [ ]:
!pip install jax[cuda12]==0.6.0

Install jaxion

In [ ]:
!pip install jaxion

## 1. Parameters

In [1]:
import jax.numpy as jnp
import jaxion

# switch on for double precision
# jax.config.update("jax_enable_x64", True)

# Parameters added/changed from default values
params = {
    "domain": {
        "resolution_multiplier": 2,
    },
    "time": {
        "end": 1.0,
    },
    "output": {
        "save": True,
    },
}

## 2. Initial Conditions

In [2]:
# Initialize the simulation
sim = jaxion.Simulation(params)

# Set initial conditions (two solitons with opposite phase)
m_22 = sim.params["quantum"]["m_22"]
box_size = sim.params["domain"]["box_size"]
r_soliton = 0.02 * box_size
xx, yy, zz = sim.grid


def rho_soliton(r, r_soliton, m_22):
    return 1.9e7 * m_22**-2 * r_soliton**-4 / (1.0 + 0.091 * (r / r_soliton) ** 2) ** 8


rho = 0.0
r_soliton = 0.02 * box_size
for i in range(2):
    x_soliton = (0.5 + 0.3 * (i - 0.5)) * box_size
    y_soliton = 0.5 * box_size
    z_soliton = 0.5 * box_size
    r = jnp.sqrt((xx - x_soliton) ** 2 + (yy - y_soliton) ** 2 + (zz - z_soliton) ** 2)
    rho += rho_soliton(r, r_soliton, m_22)

half = (xx - 0.5 * box_size) > 0
sim.state["psi"] = jnp.array(jnp.sqrt(rho) + 0j) * (1.0 * half + -1.0 * (1.0 - half))

Simulation parameters:
{
  "physics": {
    "quantum": true,
    "gravity": true,
    "hydro": false,
    "particles": false,
    "cosmology": false,
    "external_potential": false
  },
  "domain": {
    "box_size": 10.0,
    "resolution_base": 32,
    "resolution_multiplier": 2
  },
  "time": {
    "start": 0.0,
    "end": 1.0,
    "safety_factor": 1.0,
    "adaptive": false
  },
  "output": {
    "path": "./checkpoints",
    "num_checkpoints": 100,
    "save": true,
    "plot_dynamic_range": 100.0
  },
  "quantum": {
    "m_22": 1.0,
    "f_15": 0.0
  },
  "hydro": {
    "sound_speed": 1.0
  },
  "particles": {
    "num_particles": 0,
    "particle_mass": 1.0,
    "accrete_gas": false
  },
  "cosmology": {
    "omega_matter": 0.3,
    "omega_lambda": 0.7,
    "little_h": 0.7
  },
  "version": "0.0.9.post5+git.dfab348f.dirty"
}


## 3. Run!

In [3]:
# Run the simulation
sim.run()

Starting simulation (res=64, nt=4700) ...
1.0%: mcups=29.4, estimated time left (s): 41.4
2.0%: mcups=31.3, estimated time left (s): 38.6
3.0%: mcups=32.1, estimated time left (s): 37.3
4.0%: mcups=32.4, estimated time left (s): 36.5
5.0%: mcups=32.6, estimated time left (s): 35.9
6.0%: mcups=32.7, estimated time left (s): 35.4
7.0%: mcups=32.9, estimated time left (s): 34.8
8.0%: mcups=33.0, estimated time left (s): 34.3
9.0%: mcups=33.0, estimated time left (s): 33.9
10.0%: mcups=33.1, estimated time left (s): 33.5
11.0%: mcups=33.2, estimated time left (s): 33.1
12.0%: mcups=33.2, estimated time left (s): 32.7
13.0%: mcups=33.2, estimated time left (s): 32.3
14.0%: mcups=33.3, estimated time left (s): 31.9
15.0%: mcups=33.2, estimated time left (s): 31.5
16.0%: mcups=33.2, estimated time left (s): 31.1
17.0%: mcups=33.3, estimated time left (s): 30.7
18.0%: mcups=33.3, estimated time left (s): 30.4
19.0%: mcups=33.3, estimated time left (s): 30.0
20.0%: mcups=33.3, estimated time le

## 4. Visualize

In [4]:
# animate the checkpoints/*.png files
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import glob
from IPython.display import HTML

fig = plt.figure(figsize=(4, 4))
ax = plt.gca()
ax.axis("off")
imgs = []
for filename in sorted(glob.glob("checkpoints/*.png")):
    img = plt.imread(filename)
    ax = plt.gca()
    im = ax.imshow(img)
    imgs.append([im])
    plt.tight_layout(pad=0)
ani = animation.ArtistAnimation(fig, imgs, interval=50, blit=True)
plt.close(fig)
HTML(ani.to_html5_video())